## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **3d376e98-37df-4d05-bdce-a98bb6c40983**


## Dataset

# Adresses de bâtiment


## Description

L’adresse est le procédé de géolocalisation le plus fréquemment utilisé dans la vie quotidienne comme dans le domaine économique. Dans le répertoire officiel des adresses de bâtiments, chaque entrée possède un identificateur univoque et est pourvue d’une adresse elle aussi univoque. L&#x27;adresse d&#x27;un bâtiment sert communément à localiser celui-ci et permet notamment de faire le lien entre les personnes &#40;physiques ou morales&#41; et les bâtiments. Il existe pour chaque bâtiment au moins une entrée de bâtiment. En règle générale, un bâtiment compte une seule entrée desservie au niveau postal. Dans certains cas toutefois, un bâtiment &#40;par exemple bâtiments d&#x27;angle, maisons en terrasses&#41; peut compter plusieurs entrées desservies au niveau postal pour chacun des logements, des commerces, bureaux etc. L&#x27;ensemble des adresses du canton correspond au répertoire officiel des adresses de bâtiments. Il contient toutes les adresses de bâtiments postales et officielles qui sont contraignantes pour les autorités. Tous les bâtiments doivent avoir une ou plusieurs adresses univoques. Elle se compose entre autres des éléments suivants : - Le nom de la voie &#40;rue, chemin, avenue, place, etc.&#41; - Le numéro de l&#x27;adresse &#40;numéro attribué au bâtiment le long de cette voie&#41; - La localité et le code postal Son rôle est multiple : - Permettre aux services d&#x27;urgence &#40;pompiers, police, ambulances&#41; de localiser rapidement une intervention - Assurer la distribution du courrier et des colis - Servir de référence pour les actes administratifs et juridiques &#40;contrats, registres, permis, etc.&#41; - Faciliter la navigation pour tout usager &#40;livreurs, taxis, particuliers, etc.&#41; Dans le contexte suisse et genevois, l&#x27;adresse de bâtiment est rattachée au registre fédéral des bâtiments et des logements &#40;RegBL&#41;, qui recense l&#x27;ensemble des adresses officielles du pays. Chaque adresse y est géoréférencée, c&#x27;est-à-dire associée à des coordonnées géographiques précises. Une adresse est dite officielle lorsqu&#x27;elle a été validée par l&#x27;autorité compétente, en l&#x27;occurrence, dans le canton de Genève, par la Direction de l&#x27;information du territoire. Signalisation des adresses postales sur les bâtiments, une plaque de numérotation de l&#x27;adresse doit être posée de façon visible au-dessus ou à droite de la porte d&#x27;entrée des bâtiments. Celle-ci doit être maintenue en bon état. Les adresses non-postales référencent des bâtiments sans lien avec une distribution de courrier, par exemple, garage privé, poulailler, dépendance sont diponibles dans le jeu de données: Adresses non-postales.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/adresses-georeferencees-a-lentree-du-batiment)


## Metadata

- **identifier:** CAD&#95;ADRESSE@canton-geneve
- **publisher.name:** Direction de l&#x27;information du territoire
- **organization.display&#95;name:** Kanton Genf
- **organization.url:** 
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** entree-numero-municipale-adressage-localisation
- **issued:** 2025-06-12 00:00:00
- **metadata&#95;created:** 2026-01-12T16:00:09.532086
- **metadata&#95;modified:** 2026-11-16T03:05:51.688557
- **Resource id:** fd4a778e-d5f3-483b-85a0-d613a3353fec
- **Resource name:** CSV
- **Resource package&#95;id:** 3d376e98-37df-4d05-bdce-a98bb6c40983
- **Resource description:** Service téléchargement extracteur
- **Resource issued:** 2025-06-12 00:00:00
- **Resource modified:** 2026-11-14 00:00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;open
- **Resource rights:** https://opendata.swiss/terms-of-use&#35;terms&#95;open


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                # Detect file compression after HTTP compression has been decoded.
                if kwargs.get("compression", "infer") == "infer" and source.peek(2).startswith(
                    b"\x1f\x8b"
                ):
                    kwargs["compression"] = "gzip"
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **CSV** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://extract.sitg.ge.ch/files/CAD_ADRESSE-csv.zip")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** vincent.galley@etat.ge.ch &#124; Direction de l&#x27;information du territoire
